##### Boilerplate

In [1]:
# @title
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/DumLLM"

os.chdir(PROJECT_DIR)

sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

print("DumLLM initialized!")
print("Working directory:", os.getcwd())
print("Project files:", os.listdir(PROJECT_DIR))

Mounted at /content/drive
DumLLM initialized!
Working directory: /content/drive/MyDrive/DumLLM
Project files: ['requirements.txt', '.venv', '.git', 'notebooks', 'public', '.gitignore', 'start_DumLLM.ipynb']


## 3.3 Attending to different parts of the input with self-attention

### 3.3.1 A simple self-attention mechanism without trainable weights

In [3]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [6]:
input_query = inputs[1] # (x^2)
input_1 = inputs[0]

In [27]:
res = 0.
i = 0
for idx,el in enumerate(inputs[i]):
  res += inputs[i][idx] * input_query[idx]

print(res)

tensor(0.9544)


In [23]:
i = 0

res = torch.dot(inputs[i],input_query)
res

tensor(0.9544)

In [25]:
# dot product for all tokens with query
query = inputs[1]

attn_scores_2 = torch.empty(inputs.shape[0])
for i,x_i in enumerate(inputs):
  attn_scores_2[i] = torch.dot(x_i,query)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [29]:
# normalizing with softmax
def softmax_naive(x):
  return torch.exp(x) / torch.exp(x).sum()

attn_scores_2_softmax = softmax_naive(attn_scores_2)
print(attn_scores_2_softmax)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


In [30]:
# more optimized version
attn_scores_2_softmax = torch.softmax(attn_scores_2,dim=0)
print(attn_scores_2_softmax)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


In [33]:
# weighted sum with the attention scores
query = inputs[1]

context_vec_2 = torch.zeros(query.shape)

for i,x_i in enumerate(inputs):
  context_vec_2 += attn_scores_2_softmax[i] * x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


### 3.3.2 Computing attention weights for all input tokens

In [36]:
attn_scores = torch.empty(inputs.shape[0], inputs.shape[0])

for i,x_i in enumerate(inputs):
  for j,x_j in enumerate(inputs):
    attn_scores[i, j] = torch.dot(x_i,x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [38]:
# matrix multiplication
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [39]:
# normaliszing
attn_weights = torch.softmax(attn_scores,dim=1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [41]:
# all at once
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores,dim=1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [42]:
# compute context vectors
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


## 3.4 Implementing self-attention with trainable weights

### 3.4.1 Computing the attention weights step by step

In [48]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [46]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in,d_out))
W_key = torch.nn.Parameter(torch.rand(d_in,d_out))
W_value = torch.nn.Parameter(torch.rand(d_in,d_out))

In [58]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

In [59]:
print(query_2)
print(key_2)
print(value_2)

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)
tensor([0.4433, 1.1419], grad_fn=<SqueezeBackward4>)
tensor([0.3951, 1.0037], grad_fn=<SqueezeBackward4>)


In [70]:
keys = inputs @ W_key
values = inputs @ W_value

In [61]:
# dot product between Query and Key for Attention Score"
attn_scores_22 = torch.dot(query_2,key_2)
print(attn_scores_22)

tensor(1.8524, grad_fn=<DotBackward0>)


In [62]:
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)


In [67]:
# normalizing as per the paper
d_k = keys.shape[1]

attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
attn_weights_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)

In [71]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)


### 3.4.2 Implementing a compact SelfAttention Class

In [75]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
  def __init__(self, d_in, d_out):
    super().__init__()
    self.W_query = nn.Parameter(torch.rand(d_in,d_out))
    self.W_key = nn.Parameter(torch.rand(d_in,d_out))
    self.W_value = nn.Parameter(torch.rand(d_in,d_out))

  def forward(self, x):
    # attention score computing
    query = inputs @ self.W_query
    keys = inputs @ self.W_key
    values = inputs @ self.W_value

    attn_scores = query @ keys.T

    # normalizing
    d_k = keys.shape[1]
    attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)

    # context vector computing
    context_vec = attn_weights @ values
    return context_vec

torch.manual_seed(123)
self_attn = SelfAttention_v1(d_in, d_out)
self_attn(inputs)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [77]:
class SelfAttention_v2(nn.Module):
  def __init__(self, d_in, d_out, qkv_bias=False):
    super().__init__()
    self.W_query = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_key = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_value = nn.Linear(d_in,d_out,bias=qkv_bias)

  def forward(self, x):
    # attention score computing
    query = self.W_query(inputs)
    keys = self.W_key(inputs)
    values = self.W_value(inputs)

    attn_scores = query @ keys.T

    # normalizing
    d_k = keys.shape[1]
    attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)

    # context vector computing
    context_vec = attn_weights @ values
    return context_vec

torch.manual_seed(789)
self_attn = SelfAttention_v2(d_in, d_out)
self_attn(inputs)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)

## 3.5 Hiding future words with causal attention

### 3.5.1 Applying a causal attention mask

In [78]:
query = self_attn.W_query(inputs)
keys = self_attn.W_key(inputs)
values = self_attn.W_value(inputs)

attn_scores = query @ keys.T
d_k = keys.shape[1]
attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)

In [79]:
attn_weights

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)

In [80]:
# creating a mask to hide the future words
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
mask_simple

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [83]:
masked_simple = attn_weights * mask_simple
masked_simple

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)

In [84]:
# making the sum to 1 for optimizing purpose
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
masked_simple_norm

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)

In [89]:
# other way around (mask with -inf and then softmax)
mask = torch.triu(torch.ones(context_length, context_length),diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
attn_weights = torch.softmax(masked / d_k**0.5, dim=-1)
attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)

### 3.5.2 Masking additional attention weights with dropout

In [90]:
torch.manual_seed(123)

layer = torch.nn.Dropout(0.5)

In [91]:
example = torch.ones(6, 6)
example

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [92]:
layer(example)

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])

In [93]:
layer(attn_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.4925, 0.4638, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3941, 0.0000],
        [0.3869, 0.3327, 0.0000, 0.3084, 0.3331, 0.3058]],
       grad_fn=<MulBackward0>)

### 3.5.3 Implement a compact causal self-attention class

In [95]:
batch = torch.stack((inputs,inputs),dim=0)
batch

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])

In [97]:
class CausalAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
    super().__init__()
    self.W_query = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_key = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_value = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.dropout = nn.Dropout(dropout)
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length),diagonal=1))          # as a buffer

  def forward(self, x):
    b, num_tokens, d_in = x.shape

    queries = self.W_query(x)
    keys = self.W_key(x)
    values = self.W_value(x)

    attn_scores = queries @ keys.transpose(1,2)
    attn_scores.masked_fill_(
        self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
    attn_weights = torch.softmax(attn_scores / (keys.shape[1])**0.5, dim=-1
    )
    attn_weights = self.dropout(attn_weights)
    # context vector computing
    context_vec = attn_weights @ values
    return context_vec

torch.manual_seed(789)
dropout = 0
context_length = batch.shape[1]
causal_attn = CausalAttention(d_in, d_out, context_length, dropout)
causal_attn(batch)

tensor([[[-0.0872,  0.0286],
         [-0.0996,  0.0511],
         [-0.1004,  0.0645],
         [-0.0984,  0.0489],
         [-0.0512,  0.1102],
         [-0.0761,  0.0682]],

        [[-0.0872,  0.0286],
         [-0.0996,  0.0511],
         [-0.1004,  0.0645],
         [-0.0984,  0.0489],
         [-0.0512,  0.1102],
         [-0.0761,  0.0682]]], grad_fn=<UnsafeViewBackward0>)